# Gestión de pedidos online con Kafka y HDFS

Flujo: **HDFS (entrada) → Productor Python → Kafka → Consumidor Python → HDFS (salida)**

## 1. Instalar e importar librerías

In [ ]:
%pip install confluent-kafka hdfs

## Configuración

In [ ]:
import json
import socket
import time

from confluent_kafka import Producer, Consumer
from hdfs import InsecureClient


USUARIO = "iabd_user1"        


BROKER   = "10.22.87.215:9092"
HDFS_URL = "http://10.22.87.215:50070"

TOPIC    = f"pedidos_{USUARIO}"
GROUP_ID = f"grupo_pedidos_{USUARIO}_01"

RUTA_ENTRADA = f"/user/{USUARIO}/pedidos/entrada"
RUTA_SALIDA  = f"/user/{USUARIO}/pedidos/salida"
FICHERO_ENTRADA = f"{RUTA_ENTRADA}/pedidos.json"
FICHERO_SALIDA  = f"{RUTA_SALIDA}/pedidos_recibidos.json"

client = InsecureClient(HDFS_URL, user=USUARIO)
print("Topic:", TOPIC, "| Grupo:", GROUP_ID)

## Actividad 1. Preparación de los datos

In [ ]:
pedidos = [
    {"id_pedido": 1,  "cliente": "Ane",    "producto": "Portátil",          "categoria": "Informática", "cantidad": 1, "precio_unitario": 850,  "fecha": "2026-10-19"},
    {"id_pedido": 2,  "cliente": "Jon",    "producto": "Ratón",             "categoria": "Accesorios",  "cantidad": 2, "precio_unitario": 25,   "fecha": "2026-10-19"},
    {"id_pedido": 3,  "cliente": "Maite",  "producto": "Teclado mecánico",  "categoria": "Accesorios",  "cantidad": 1, "precio_unitario": 70,   "fecha": "2026-10-19"},
    {"id_pedido": 4,  "cliente": "Unai",   "producto": "Monitor 27\"",      "categoria": "Informática", "cantidad": 2, "precio_unitario": 210,  "fecha": "2026-10-19"},
    {"id_pedido": 5,  "cliente": "Leire",  "producto": "Auriculares",       "categoria": "Audio",       "cantidad": 1, "precio_unitario": 60,   "fecha": "2026-10-19"},
    {"id_pedido": 6,  "cliente": "Iker",   "producto": "Smartphone",        "categoria": "Móviles",     "cantidad": 1, "precio_unitario": 599,  "fecha": "2026-10-19"},
    {"id_pedido": 7,  "cliente": "Amaia",  "producto": "Funda móvil",       "categoria": "Accesorios",  "cantidad": 3, "precio_unitario": 12,   "fecha": "2026-10-19"},
    {"id_pedido": 8,  "cliente": "Mikel",  "producto": "Tablet",            "categoria": "Móviles",     "cantidad": 1, "precio_unitario": 320,  "fecha": "2026-10-19"},
    {"id_pedido": 9,  "cliente": "Nerea",  "producto": "Altavoz Bluetooth", "categoria": "Audio",       "cantidad": 2, "precio_unitario": 45,   "fecha": "2026-10-19"},
    {"id_pedido": 10, "cliente": "Asier",  "producto": "Disco SSD 1TB",     "categoria": "Informática", "cantidad": 2, "precio_unitario": 95,   "fecha": "2026-10-19"},
    {"id_pedido": 11, "cliente": "Olatz",  "producto": "Webcam HD",         "categoria": "Accesorios",  "cantidad": 1, "precio_unitario": 40,   "fecha": "2026-10-19"},
    {"id_pedido": 12, "cliente": "Gorka",  "producto": "Impresora",         "categoria": "Informática", "cantidad": 1, "precio_unitario": 130,  "fecha": "2026-10-20"},
    {"id_pedido": 13, "cliente": "Itziar", "producto": "Cargador USB-C",    "categoria": "Accesorios",  "cantidad": 4, "precio_unitario": 18,   "fecha": "2026-10-20"},
    {"id_pedido": 14, "cliente": "Aitor",  "producto": "Smartwatch",        "categoria": "Móviles",     "cantidad": 1, "precio_unitario": 180,  "fecha": "2026-10-20"},
    {"id_pedido": 15, "cliente": "Ainhoa", "producto": "Barra de sonido",   "categoria": "Audio",       "cantidad": 1, "precio_unitario": 150,  "fecha": "2026-10-20"},
    {"id_pedido": 16, "cliente": "Xabier", "producto": "Router WiFi 6",     "categoria": "Redes",       "cantidad": 1, "precio_unitario": 110,  "fecha": "2026-10-20"},
    {"id_pedido": 17, "cliente": "Maialen","producto": "Cable HDMI",        "categoria": "Accesorios",  "cantidad": 5, "precio_unitario": 9,    "fecha": "2026-10-20"},
    {"id_pedido": 18, "cliente": "Josu",   "producto": "Portátil gaming",   "categoria": "Informática", "cantidad": 1, "precio_unitario": 1450, "fecha": "2026-10-20"},
    {"id_pedido": 19, "cliente": "Garazi", "producto": "Switch de red",     "categoria": "Redes",       "cantidad": 2, "precio_unitario": 35,   "fecha": "2026-10-20"},
    {"id_pedido": 20, "cliente": "Oier",   "producto": "Micrófono USB",     "categoria": "Audio",       "cantidad": 1, "precio_unitario": 85,   "fecha": "2026-10-20"},
    {"id_pedido": 21, "cliente": "Naroa",  "producto": "Teclado inalámbrico","categoria": "Accesorios", "cantidad": 2, "precio_unitario": 38,   "fecha": "2026-10-20"},
    {"id_pedido": 22, "cliente": "Beñat",  "producto": "Smartphone",        "categoria": "Móviles",     "cantidad": 2, "precio_unitario": 450,  "fecha": "2026-10-20"},
    {"id_pedido": 23, "cliente": "Uxue",   "producto": "Repetidor WiFi",    "categoria": "Redes",       "cantidad": 3, "precio_unitario": 29,   "fecha": "2026-10-20"},
    {"id_pedido": 24, "cliente": "Eneko",  "producto": "Monitor 24\"",      "categoria": "Informática", "cantidad": 1, "precio_unitario": 140,  "fecha": "2026-10-20"},
]

# Comprobaciones básicas: más de 20 pedidos y ids únicos
ids = [p["id_pedido"] for p in pedidos]
assert len(pedidos) >= 20, "Se necesitan al menos 20 pedidos"
assert len(ids) == len(set(ids)), "Hay id_pedido repetidos"
print(f"{len(pedidos)} pedidos preparados")

In [ ]:
# Guardar copia local pedidos.json
with open("pedidos.json", "w", encoding="utf-8") as f:
    json.dump(pedidos, f, ensure_ascii=False, indent=2)

# Crear carpetas en HDFS y subir el archivo a la carpeta de entrada
client.makedirs(RUTA_ENTRADA)
client.makedirs(RUTA_SALIDA)

client.upload(FICHERO_ENTRADA, "pedidos.json", overwrite=True)

print("Contenido de", RUTA_ENTRADA, "->", client.list(RUTA_ENTRADA))

## Actividad 2. Productor Kafka

In [ ]:
conf_productor = {
    "bootstrap.servers": BROKER,
    "client.id": socket.gethostname()
}
producer = Producer(conf_productor)

# Leer pedidos.json DIRECTAMENTE desde HDFS
with client.read(FICHERO_ENTRADA, encoding="utf-8") as reader:
    pedidos_hdfs = json.load(reader)

print(f"Pedidos leídos de HDFS: {len(pedidos_hdfs)}")

In [ ]:
entregados = 0
errores = 0

def confirmar(err, msg):
    global entregados, errores
    if err:
        errores += 1
        print("Error al enviar:", err)
    else:
        entregados += 1
        pedido = json.loads(msg.value().decode("utf-8"))
        print(f"Enviado pedido {pedido['id_pedido']} -> "
              f"{msg.topic()} [partición {msg.partition()}, offset {msg.offset()}]")

for pedido in pedidos_hdfs:
    producer.produce(
        TOPIC,
        key=str(pedido["id_pedido"]),                      # clave = id_pedido
        value=json.dumps(pedido, ensure_ascii=False),      # un mensaje JSON por pedido
        callback=confirmar
    )
    producer.poll(0)   # atiende callbacks pendientes

pendientes = producer.flush(15)
print(f"\nEntregados: {entregados} | Errores: {errores} | Sin confirmar: {pendientes}")

## Actividad 3. Consumidor Kafka

In [ ]:
conf_consumidor = {
    "bootstrap.servers": BROKER,
    "group.id": GROUP_ID,                 # grupo propio
    "auto.offset.reset": "earliest"       # leer desde el principio del tópico
}
consumer = Consumer(conf_consumidor)
consumer.subscribe([TOPIC])

ids_esperados = {p["id_pedido"] for p in pedidos_hdfs}
TOTAL_ESPERADO = len(ids_esperados)
TIEMPO_MAXIMO = 60          # segundos de espera máxima en total

pedidos_recibidos = []      # lista de diccionarios
ids_recibidos = set()
duplicados = 0
mensajes_invalidos = 0

inicio = time.monotonic()
try:
    while len(ids_recibidos) < TOTAL_ESPERADO:
        if time.monotonic() - inicio > TIEMPO_MAXIMO:
            print(f"Tiempo máximo de espera ({TIEMPO_MAXIMO}s) alcanzado")
            break

        msg = consumer.poll(timeout=2.0)

        if msg is None:
            print("No hay mensajes")
            continue

        if msg.error():
            print("Error:", msg.error())
            continue

        try:
            pedido = json.loads(msg.value().decode("utf-8"))   # JSON -> dict
        except (json.JSONDecodeError, UnicodeDecodeError) as e:
            mensajes_invalidos += 1
            print("Mensaje no válido:", e)
            continue

        id_p = pedido["id_pedido"]
        if id_p in ids_recibidos:
            # Puede pasar si el productor se ejecutó varias veces sobre el mismo tópico
            duplicados += 1
            print(f"Duplicado ignorado: pedido {id_p}")
            continue

        ids_recibidos.add(id_p)
        pedidos_recibidos.append(pedido)
        print("Recibido:", pedido)
finally:
    consumer.close()

print(f"\nRecibidos: {len(pedidos_recibidos)} de {TOTAL_ESPERADO} | "
      f"Duplicados ignorados: {duplicados} | Inválidos: {mensajes_invalidos}")
print("¿Se han recuperado todos los pedidos?", ids_recibidos == ids_esperados)

## Actividad 4. Escritura en HDFS

In [ ]:
# Comprobar pérdidas y duplicados antes de escribir
faltan = ids_esperados - ids_recibidos
sobran = ids_recibidos - ids_esperados
ids_lista = [p["id_pedido"] for p in pedidos_recibidos]

print("Faltan:", sorted(faltan))
print("Sobran:", sorted(sobran))
print("Duplicados en la lista:", len(ids_lista) != len(set(ids_lista)))

assert not faltan and not sobran, "No coinciden los identificadores: no se escribe el archivo"
assert len(ids_lista) == len(set(ids_lista)), "Hay duplicados en la lista"

In [ ]:
# Ordenado por id para que el resultado sea predecible
pedidos_recibidos.sort(key=lambda p: p["id_pedido"])
contenido = json.dumps(pedidos_recibidos, ensure_ascii=False, indent=2)

client.write(FICHERO_SALIDA, data=contenido, encoding="utf-8", overwrite=True)

# Verificar la escritura: el tamaño en HDFS debe coincidir con lo escrito
tam_hdfs = client.status(FICHERO_SALIDA)["length"]
tam_local = len(contenido.encode("utf-8"))
print(f"Tamaño en HDFS: {tam_hdfs} bytes | Tamaño escrito: {tam_local} bytes")
assert tam_hdfs == tam_local, "El tamaño en HDFS no coincide"

print("Contenido de", RUTA_SALIDA, "->", client.list(RUTA_SALIDA))

## Actividad 5. Lectura y comprobación desde HDFS

In [ ]:
# Leer pedidos_recibidos.json directamente desde HDFS (sin Pandas)
with client.read(FICHERO_SALIDA, encoding="utf-8") as reader:
    datos = json.load(reader)

# 1. Número total de pedidos
total_pedidos = len(datos)
print("Total de pedidos almacenados:", total_pedidos)

# 2. Importe total (cantidad x precio_unitario)
importe_total = sum(p["cantidad"] * p["precio_unitario"] for p in datos)
print("Importe total:", importe_total, "€")

# 3. Pedido con mayor importe
pedido_max = max(datos, key=lambda p: p["cantidad"] * p["precio_unitario"])
print("Pedido con mayor importe:")
print(f"  id {pedido_max['id_pedido']} - {pedido_max['cliente']} - {pedido_max['producto']} "
      f"-> {pedido_max['cantidad'] * pedido_max['precio_unitario']} €")

# 4. Pedidos por categoría
por_categoria = {}
for p in datos:
    por_categoria[p["categoria"]] = por_categoria.get(p["categoria"], 0) + 1
print("Pedidos por categoría:")
for categoria, n in sorted(por_categoria.items()):
    print(f"  {categoria}: {n}")

# 5. Todos los identificadores originales aparecen en el archivo final
ids_finales = {p["id_pedido"] for p in datos}
ids_originales = {p["id_pedido"] for p in pedidos}
faltan_final = ids_originales - ids_finales
print("\nIdentificadores originales que faltan en el archivo final:", sorted(faltan_final))
print("¿Todos los identificadores originales están en el archivo final?", not faltan_final)